In [2]:
import json, pathlib, requests
from urllib.parse import urljoin

In [3]:
# read the animation manifest
j = requests.get("https://services.swpc.noaa.gov/products/animations/geospace/pressure.json").json()

# pick a few frames for testing
frames = j[:10]  # or filter by time_tag

out = pathlib.Path('swpc_pressure')
(out / 'png').mkdir(parents=True, exist_ok=True)
(out / 'io2').mkdir(parents=True, exist_ok=True)

In [6]:
for f in j:
    # download PNG
    png_url = urljoin("https://services.swpc.noaa.gov", f["url"])
    png_path = out / "png" / png_url.split("/")[-1]
    png_path.write_bytes(requests.get(png_url, timeout=30).content)

    # find corresponding NOMADS date
    day = f["time_tag"][:10].replace("-", "")  # YYYYMMDD
    base = f"https://nomads.ncep.noaa.gov/pub/data/nccf/com/swmf/prod/swmf.{day}/GM/IO2/"

    # naive: try to pull the equatorial and meridional files that bracket the frame minute
    # in practice, list the directory and choose y0_..._END.txt and z0_..._END.txt whose END ~= time_tag
    # example filenames pattern:
    # y0_YYYYMMDDTHHMM_YYYYMMDDTHHMM  and z0_YYYYMMDDTHHMM_YYYYMMDDTHHMM
    # you can scrape the index and match the trailing time